# 19 — Multi-Seed + Selang Kepercayaan (CI) untuk MCC utama
**Pertanyaan (Reviewer K4):** laporkan CI/std untuk seluruh nilai MCC dan jalankan
eksperimen (khususnya few-shot) dengan beberapa *random seed*.

**Rancangan (jujur, angka nyata):** ulang eksperimen *offline* utama pada
**5 seed** {42,1,7,123,2024} dan laporkan **mean +/- std** serta **CI 95%**
(t-interval, df=n-1). Menutup tiga tabel MCC utama naskah:
- `tab:baseline` (cross-network baseline: CIC->UNSW, UNSW->CIC)
- `tab:align` (CORAL + joint training)
- `tab:fewshot` (few-shot k=1/5/10%)

Seluruh komponen stokastik divariasikan per seed (split CIC train/test, subsample joint,
sampling few-shot, random_state XGBoost) agar CI benar-benar mencerminkan variasi,
bukan CI palsu. Protokol/mapping identik nb05/nb06 (Model A biner, z-score per dataset,
tanpa kebocoran). UNSW memakai split rilis tetap.
> Jalankan di **SageMaker** (butuh `cleaned_100.pkl` + CSV UNSW). Waktu jalan lebih lama
> (5x eksperimen); wajar.

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('scipy','scipy')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import pickle, os, json, datetime
import numpy as np, pandas as pd
from scipy.linalg import sqrtm
from scipy import stats as sps
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef
from xgboost import XGBClassifier
CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../multiseed_ci.json'
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); S3_PREFIX='unsw-far'
SEEDS=[42,1,7,123,2024]; FRACS=[0.0,0.01,0.05,0.10]
print('CIC:', os.path.exists(CIC_PKL), '| seeds:', SEEDS)
print('=== SEL 1 SELESAI ===')

In [ ]:
# --- Mapping & util (identik nb05/06) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())
def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan).fillna(0.0)
    return out.astype(float).values
def make_xgb(seed):
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=seed,tree_method='hist')
def mcc(yt,yp): return float(matthews_corrcoef(yt,yp))
def coral(Xs,Xt,eps=1.0):
    d=Xs.shape[1]
    Cs=np.cov(Xs,rowvar=False)+eps*np.eye(d); Ct=np.cov(Xt,rowvar=False)+eps*np.eye(d)
    return Xs @ np.real(sqrtm(np.linalg.inv(Cs))) @ np.real(sqrtm(Ct))
print('=== SEL 2 SELESAI ===')

In [ ]:
# --- Muat data mentah sekali (UNSW split rilis tetap; CIC di-split per seed) ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)
Xc_all=build_matrix(cic_df,'cic')
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC all:',Xc_all.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)
print('=== SEL 3 SELESAI ===')

In [ ]:
# --- Satu seed = satu replikasi lengkap semua eksperimen offline ---
def run_seed(seed):
    r={}
    # split CIC per seed + z-score (fit train saja)
    Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,
                                                     random_state=seed,stratify=y_cic)
    scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
    # baseline single-source (cross)
    m=make_xgb(seed); m.fit(Xc_tr,yc_tr); r['base_cic2unsw']=mcc(y_ute,m.predict(Xu_te))
    m=make_xgb(seed); m.fit(Xu_tr,y_utr); r['base_unsw2cic']=mcc(yc_te,m.predict(Xc_te))
    # CORAL
    m=make_xgb(seed); m.fit(coral(Xc_tr,Xu_tr),yc_tr); r['coral_cic2unsw']=mcc(y_ute,m.predict(Xu_te))
    m=make_xgb(seed); m.fit(coral(Xu_tr,Xc_tr),y_utr); r['coral_unsw2cic']=mcc(yc_te,m.predict(Xc_te))
    # joint (subsample CIC ~3:1 per seed)
    rng=np.random.RandomState(seed); n_u=len(Xu_tr)
    ci=rng.choice(len(Xc_tr),min(len(Xc_tr),n_u*3),replace=False)
    Xj=np.vstack([Xc_tr[ci],Xu_tr]); yj=np.concatenate([yc_tr[ci],y_utr])
    mj=make_xgb(seed); mj.fit(Xj,yj)
    r['joint_cic']=mcc(yc_te,mj.predict(Xc_te)); r['joint_unsw']=mcc(y_ute,mj.predict(Xu_te))
    # few-shot dua arah
    def fs(Xs,ys,Xt_tr,yt_tr,Xt_te,yt_te,fr):
        if fr==0.0: Xtr,ytr=Xs,ys
        else:
            rr=np.random.RandomState(seed); n=int(len(Xt_tr)*fr)
            idx=rr.choice(len(Xt_tr),n,replace=False)
            Xtr=np.vstack([Xs,Xt_tr[idx]]); ytr=np.concatenate([ys,yt_tr[idx]])
        mm=make_xgb(seed); mm.fit(Xtr,ytr); return mcc(yt_te,mm.predict(Xt_te))
    for fr in FRACS:
        r[f'fs_cic2unsw_{fr}']=fs(Xc_tr,yc_tr,Xu_tr,y_utr,Xu_te,y_ute,fr)
        r[f'fs_unsw2cic_{fr}']=fs(Xu_tr,y_utr,Xc_tr,yc_tr,Xc_te,yc_te,fr)
    return r

all_runs={}
for sd in SEEDS:
    all_runs[sd]=run_seed(sd)
    print(f'seed {sd}: base c2u={all_runs[sd]["base_cic2unsw"]:+.3f} '
          f'fs1% c2u={all_runs[sd]["fs_cic2unsw_0.01"]:.3f} u2c={all_runs[sd]["fs_unsw2cic_0.01"]:.3f} '
          f'joint(C/U)={all_runs[sd]["joint_cic"]:.3f}/{all_runs[sd]["joint_unsw"]:.3f}')
print('=== SEL 4 SELESAI ===')

In [ ]:
# --- Agregasi: mean +/- std + CI 95% (t-interval) ---
def agg(key):
    v=np.array([all_runs[sd][key] for sd in SEEDS],float)
    m=v.mean(); sd=v.std(ddof=1); n=len(v)
    h=sps.t.ppf(0.975,n-1)*sd/np.sqrt(n) if n>1 else 0.0
    return dict(mean=round(float(m),4),std=round(float(sd),4),
                ci95=[round(float(m-h),4),round(float(m+h),4)],
                values=[round(float(x),4) for x in v])
keys=['base_cic2unsw','base_unsw2cic','coral_cic2unsw','coral_unsw2cic',
      'joint_cic','joint_unsw']+[f'fs_{d}_{fr}' for fr in FRACS for d in ('cic2unsw','unsw2cic')]
summary={k:agg(k) for k in keys}
def show(lbl,k):
    a=summary[k]; print(f'{lbl:28s} {a["mean"]:+.4f} +/- {a["std"]:.4f}  CI95 {a["ci95"]}')
print('== Baseline cross =='); show('CIC->UNSW',"base_cic2unsw"); show('UNSW->CIC',"base_unsw2cic")
print('== CORAL =='); show('CIC->UNSW',"coral_cic2unsw"); show('UNSW->CIC',"coral_unsw2cic")
print('== Joint =='); show('uji CIC',"joint_cic"); show('uji UNSW',"joint_unsw")
print('== Few-shot CIC->UNSW =='); [show(f'  k={fr}',f'fs_cic2unsw_{fr}') for fr in FRACS]
print('== Few-shot UNSW->CIC =='); [show(f'  k={fr}',f'fs_unsw2cic_{fr}') for fr in FRACS]
print('=== SEL 5 SELESAI ===')

In [ ]:
# --- Simpan + upload S3 ---
meta=dict(deskripsi='Multi-seed (5) MCC dgn mean/std/CI95 t-interval untuk tab:baseline, tab:align, tab:fewshot. Protokol identik nb05/06; semua komponen stokastik divariasikan per seed.',
          generated=datetime.datetime.utcnow().isoformat()+'Z', seeds=SEEDS, fracs=FRACS,
          summary=summary, raw=all_runs)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('Saved:',OUT_JSON)
try:
    import boto3; s3=boto3.client('s3')
    s3.upload_file(OUT_JSON,S3_BUCKET,f'{S3_PREFIX}/multiseed/multiseed_ci.json'); print('  upload -> multiseed/multiseed_ci.json')
except Exception as e:
    print('upload S3 dilewati:',e)
print('=== SEL 6 SELESAI ===')